# Subgame-Perfect Equilibrium, Principals MDP 

## Objective of the Principal 
The principal chooses a contract $b(o) \in B$ (defined per outcome $o \in O$) to:
- induce the agent to take a desired action $a^p$
- maximize its utility $\mathbb{E}_{o \sim O(a)}[r^{p}(o) - b(o)]$

## General Formulation of the Problem 

Reward Function:
$$
R^{p}(s,b,o) = r^{p}(s,o) - b(o)
$$

At each timestep $t$ the principal observes the state $s_t$ and constructs a contract $b_t \in B$ according to its policy
$$
\rho : S \rightarrow \Delta(B)
$$

Value Function $V^\rho(\pi)$ of a policy $\rho$ given the agent’s policy $\pi$ is defined in a state $s$ by
$$
V^{\rho}(s \mid \pi) = \mathbb{E}\left[\sum_{t} \gamma^{t} R^{p}(s_t, b_t, o_t) \mid s_{0} = s\right]
$$

Q-value Function:
$$
Q^{\rho}(s, b \mid \pi) = V^{\rho}(s \mid b_{0} = b, \pi)
$$


### Principal's perspective 
The principals MDP defined by $\pi$, refer to the optimal policy $\rho^{*}(\pi)$ as subgame-perfect against $\pi$. 

In all states, this policy satisfies 
$$
\rho(s \mid \pi) \in \argmax_{b}Q^{\rho}(s, b \mid \pi) 
$$ 
where 
$$
Q^{*}(s,b \mid \pi) \equiv Q^{\rho^{*}(\pi)}(s,b \mid \pi) 
$$

## Differecnce to Learning Setting 
The principal does not learn the action $a^{p}$ that he wants the agent to take

In [ ]:
class PrincipalMeta:
    """
    Principal in a Principal-Agent MDP
    Solves outer optimization of the meta-algorithm:
    given the agents policy pi, find best responding rho 
    """ 

    def __init__(self, mdp, r_p, b_grid_step=0.1):
        self.mdp = mdp
        self.n_states = mdp.n_states
        self.n_outcomes = mdp.n_outcomes
        self.gamma = mdp.gamma
        self.r_p = r_p
        self.b_values = np.round(np.arange(0, 1.01, b_grid_step), 3)
        self.contracts = list(product(self.b_values, repeat=self.n_outcomes))
        self.n_contracts = len(self.contracts)

        self.V = None
        self.Q = None
        self.rho_star = None

    def solve(self, pi, tol=1e-4, max_iter=1000, snapshot_every=1):
        """
        Fix pi: solve principal's MDP via value iteration.
        """
        mdp = self.mdp
        V = {s: 0.0 for s in range(self.n_states)}
        snapshots = []

        for it in range(1, max_iter + 1):
            delta = 0.0
            new_V = {}

            for s in range(self.n_states):
                a = pi[s]

                q_values = []
                for b in self.contracts:
                    q = sum(
                        mdp.P_outcome[s, a, o] * (self.r_p[s, o] - b[o] + self.gamma * V[mdp.T[s, o]])
                        for o in range(self.n_outcomes)
                    )
                    q_values.append(q)

                new_V[s] = max(q_values)
                delta = max(delta, abs(new_V[s] - V[s]))

            V = new_V

            rho_star = {}
            for s in range(self.n_states):
                a = pi[s]
                q_values = [
                    sum(
                        mdp.P_outcome[s, a, o] * (self.r_p[s, o] - b[o] + self.gamma * V[mdp.T[s, o]])
                        for o in range(self.n_outcomes)
                    )
                    for b in self.contracts
                ]
                rho_star[s] = self.contracts[int(np.argmax(q_values))]

            if it % snapshot_every == 0 or delta < tol:
                snapshots.append((V.copy(), rho_star.copy(), delta, it))

            if delta < tol:
                print(f"Principal VI converged in {it} iterations (Δ={delta:.6f})")
                break

        self.V = V
        self.rho_star = rho_star

        return snapshots